# Phase 3: Data Generation and Augmentation
## 3.1: Handling Imbalanced Data with SMOTE

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings

# Suppress specific warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)

### Step 1: Prepare the Data
Make sure your 'train' variable from Phase 1 is in memory.

In [ ]:
# Define features (X) and target (y)
target = 'Class/ASD'
X = train.drop(columns=[target])
y = train[target]

print(f"Original Dataset Shape - X: {X.shape}, y: {y.shape}")
print(f"Original Class Distribution: {Counter(y)}")
print("-" * 50)

### Step 2: Strategic Train-Test Split (CRITICAL STEP)
We must split the data FIRST, and only augment the training set to prevent Data Leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Class Distribution in Training Set BEFORE SMOTE: {Counter(y_train)}")
print(f"Class Distribution in Test Set (Will NOT be augmented): {Counter(y_test)}")
print("-" * 50)

### Step 3: Apply SMOTE to the Training Data

In [ ]:
smote_sampler = SMOTE(
    sampling_strategy='auto', 
    random_state=42, 
    k_neighbors=5
)

# Fit and transform the training data ONLY
X_train_smote, y_train_smote = smote_sampler.fit_resample(X_train, y_train)

print(f"Class Distribution in Training Set AFTER SMOTE: {Counter(y_train_smote)}")
print(f"New Augmented Training Data Shape: {X_train_smote.shape}")
print("-" * 50)

### Step 4: Evaluate SMOTE Impact on XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. Train on ORIGINAL Imbalanced Data
xgb_original = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, eval_metric='logloss', random_state=42)
xgb_original.fit(X_train, y_train)
y_pred_orig = xgb_original.predict(X_test)
y_prob_orig = xgb_original.predict_proba(X_test)[:, 1]

# 2. Train on SMOTE Augmented Data
xgb_smote = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, eval_metric='logloss', random_state=42)
xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = xgb_smote.predict(X_test)
y_prob_smote = xgb_smote.predict_proba(X_test)[:, 1]

# 3. Compare Results
print("=== XGBoost Performance on ORIGINAL Imbalanced Data ===")
print(classification_report(y_test, y_pred_orig))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_orig):.4f}\n")

print("=== XGBoost Performance on SMOTE Augmented Data ===")
print(classification_report(y_test, y_pred_smote))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_smote):.4f}")